# 🚀 AI Video Dubbing Studio - Cloud GPU Server (Google Colab / Kaggle)

Welcome to the **AI Video Dubbing Cloud GPU Server**! This notebook runs the full heavy processing pipeline (Faster-Whisper, Chatterbox Multilingual / XTTS v2 Zero-Shot Voice Cloning, Wav2Lip GAN, UVR Background Separator) on a **free Cloud GPU (NVIDIA Tesla T4 / A100 / P100)**.

### 📋 Setup:
1. In Google Colab menu, go to **Runtime > Change runtime type** and select **T4 GPU** (or A100 if Colab Pro).
2. Run **Cell 1** (Install Dependencies & Download Models). Everything installs cleanly with zero restarts needed!

### Then pick ONE of these two ways to dub a video:

**Option A - Everything inside Colab, no local app needed (easiest):**
- Run **Cell 3** below. Upload a video, paste a YouTube link, or paste a Google Drive share link in the form on the right, adjust settings if you want (they default to the highest quality already), and run - it downloads/plays the finished video right here.

**Option B - Use the local Video Dubbing GUI on your own PC:**
- Run **Cell 2** (Start Cloud GPU Server & Tunnel) instead of Cell 3.
- Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`) it prints and paste it into your local **Video Dubbing GUI**'s "Remote Cloud GPU" field.

### 🔑 Groq API Key (translation) - 1 required + 4 optional fallback keys:
Both Cell 2 and Cell 3 have a `groq_api_key` field (required for AI-powered context-aware translation) plus `groq_api_key_2` through `groq_api_key_5` (optional). If the primary key hits Groq's free-tier rate limit partway through a long video, processing automatically shifts to the next key, then the next, and so on - so add a few keys from separate free Groq accounts if you're dubbing long/many videos. Leaving all of them blank falls back to Colab's local translator (no AI condensing).

### 📁 Google Drive video source:
Pick **Google Drive Link** in Cell 3's `video_source` dropdown and paste the file's **Share** link (`drive.google.com/file/d/.../view`) into `gdrive_link`. Make sure the file's sharing is set to **"Anyone with the link"**, otherwise the download will fail.

### 🔗 If a YouTube link fails to download:
Colab's IP is shared by thousands of users, so YouTube sometimes blocks downloads from it with a bot-check. This only matters if you're pasting a YouTube link (not needed for uploads or Google Drive links). If it happens:
1. Export a `cookies.txt` from a browser where you're logged into YouTube (e.g. the "Get cookies.txt LOCALLY" extension).
2. Upload it to the Colab file browser on the left (wherever you like), then paste its path (e.g. `/content/cookies.txt`) into the **YouTube Cookies File** field in Cell 3's form.
3. ⚠️ **Treat this file as secret** - it's your live logged-in session, not a reusable API key. Never commit it or upload it to GitHub (the repo's `.gitignore` already blocks any `cookies*.txt` from being committed by accident). It only needs to exist in this Colab runtime, which is wiped when the runtime restarts.

In [ ]:
# Cell 1: Check GPU & Install Dependencies
!nvidia-smi

# Always start from /content so re-running this cell (e.g. after a git pull) never
# accidentally "cd's into itself" and lands in a nested/wrong directory.
%cd /content

# Clone repo or pull latest code
!git clone https://github.com/Asadullah404/Ai_Video_Dubbing.git dubbing_app || (cd dubbing_app && git pull)
%cd /content/dubbing_app

# Install server & dubbing dependencies cleanly without dependency conflict warnings
!pip install -q --no-warn-conflicts fastapi uvicorn python-multipart pycloudflared nest-asyncio deep-translator gdown
!pip install -q --no-warn-conflicts faster-whisper coqui-tts pyannote.audio audio-separator[gpu] speechbrain groq librosa soundfile noisereduce pedalboard resemblyzer gTTS yt-dlp opencv-python

# Install Chatterbox Multilingual voice cloning engine & prerequisites (--no-deps)
!pip install -q --no-warn-conflicts --no-deps chatterbox-tts diffusers==0.29.0 resemble-perth conformer==0.3.2 s3tokenizer pykakasi==2.3.0 spacy-pkuseg pyloudnorm

# Download pre-trained Wav2Lip and S3FD weights
!mkdir -p Wav2Lip/face_detection/detection/sfd
!wget -q -c 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights2/wav2lip_gan.1.1.pth' -O 'Wav2Lip/wav2lip_gan.pth'
!wget -q -c 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights1/s3fd-619a316812.1.1.pth' -O 'Wav2Lip/face_detection/detection/sfd/s3fd.pth'

print("\n✅ All Dependencies and Pre-trained Models are installed and ready!")
print("🚀 You can now directly run Cell 2 or Cell 3 below (no session restart required).")

In [ ]:
# Cell 2: Launch Cloud GPU Server & Expose via Cloudflare Tunnel
import os

# Optional: Set your HuggingFace token for PyAnnote speaker diarization
os.environ["HF_TOKEN"] = ""

# Groq API key (required for AI-powered translation) + up to 4 optional fallback keys.
# If the primary key hits Groq's rate limit mid-video, processing automatically shifts to
# the next key, then the next, and so on. Leave the fallback fields blank if you only have one key.
groq_api_key = ""       # required
groq_api_key_2 = ""     # optional fallback
groq_api_key_3 = ""     # optional fallback
groq_api_key_4 = ""     # optional fallback
groq_api_key_5 = ""     # optional fallback

os.environ["Groq_TOKEN"] = ",".join(
    k.strip() for k in [groq_api_key, groq_api_key_2, groq_api_key_3, groq_api_key_4, groq_api_key_5]
    if k.strip()
)
os.environ["COQUI_TOS_AGREED"] = "1"

# Run server
!python colab_server.py

In [ ]:
#@title Cell 3: 🎬 One-Click Dubbing (upload a video, paste a link, or use Google Drive - right here in Colab) { display-mode: "form" }
#@markdown Fill in the form on the right, then run this cell (▶). Settings already default to the highest quality - only Target Language really needs changing.

video_source = "Upload from computer" #@param ["Upload from computer", "YouTube URL", "Google Drive Link"]
youtube_url = "" #@param {type:"string"}
youtube_cookies_path = "" #@param {type:"string"}
gdrive_link = "" #@param {type:"string"}
source_language = "en" #@param ["auto","en","es","fr","de","it","pt","pl","tr","ru","nl","cs","ar","zh-cn","ja","ko","hi","ur","hu"]
target_language = "es" #@param ["en","es","fr","de","it","pt","pl","tr","ru","nl","cs","ar","zh-cn","ja","ko","hi","ur","hu","bn","ta","te","ml","th","vi","id","ms","fa","sw","ne","si"]
whisper_model = "large-v3" #@param ["tiny","base","small","medium","large-v3"]
voice_quality = "ultra" #@param ["standard","high","ultra"]
enable_lipsync = True #@param {type:"boolean"}
preserve_background_audio = True #@param {type:"boolean"}
hf_token = "" #@param {type:"string"}
groq_api_key = "" #@param {type:"string"}
groq_api_key_2 = "" #@param {type:"string"}
groq_api_key_3 = "" #@param {type:"string"}
groq_api_key_4 = "" #@param {type:"string"}
groq_api_key_5 = "" #@param {type:"string"}

import os
import sys
import subprocess

%cd /content/dubbing_app

if hf_token.strip():
    os.environ["HF_TOKEN"] = hf_token.strip()

# 1 required Groq key + up to 4 optional fallback keys - if one hits Groq's rate limit
# mid-video, processing automatically shifts to the next key, then the next, and so on.
groq_keys = [k.strip() for k in [groq_api_key, groq_api_key_2, groq_api_key_3, groq_api_key_4, groq_api_key_5] if k.strip()]
if groq_keys:
    os.environ["Groq_TOKEN"] = ",".join(groq_keys)
    if len(groq_keys) > 1:
        print(f"🔑 {len(groq_keys)} Groq API keys loaded (auto fail-over on rate limit)")

if youtube_cookies_path.strip():
    os.environ["YT_COOKIES_FILE"] = youtube_cookies_path.strip()
os.environ["COQUI_TOS_AGREED"] = "1"

# Get the video
uploaded_video_path = ""
if video_source == "Upload from computer":
    from google.colab import files
    print("📤 Choose a video file to upload...")
    uploaded = files.upload()
    if not uploaded:
        raise Exception("No file uploaded.")
    uploaded_video_path = list(uploaded.keys())[0]
    print(f"\n🎬 Uploaded video ready: {uploaded_video_path}")
elif video_source == "Google Drive Link":
    if not gdrive_link.strip():
        raise Exception("Please paste a Google Drive share link in the form above.")
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown
    print("📥 Downloading video from Google Drive...")
    # fuzzy=True lets gdown accept a normal "Share" link (drive.google.com/file/d/FILE_ID/view)
    # instead of requiring the raw file ID. The Drive file must be shared as "Anyone with the link".
    uploaded_video_path = gdown.download(url=gdrive_link.strip(), output="gdrive_input_video.mp4",
                                          quiet=False, fuzzy=True)
    if not uploaded_video_path or not os.path.exists(uploaded_video_path):
        raise Exception("Google Drive download failed - make sure the file's sharing is set to "
                         "'Anyone with the link' and the link points to a single video file.")
    print(f"\n🎬 Downloaded video ready: {uploaded_video_path}")
else:
    if not youtube_url.strip():
        raise Exception("Please paste a YouTube URL in the form above.")

# Run the full dubbing pipeline in a clean subprocess to ensure 100% fresh C-extensions
# and zero dependency conflict / restart issues.
runner_code = f"""
import os
import sys
from video_dubbing_core import EnhancedVideoDubbing, download_youtube_video

video_source = {repr(video_source)}
youtube_url = {repr(youtube_url.strip())}
video_path = {repr(uploaded_video_path)}

if video_source == "YouTube URL":
    print("📥 Downloading YouTube video...")
    video_path = download_youtube_video(youtube_url)
    if not video_path:
        print("❌ YouTube download failed - check the URL or cookies.", file=sys.stderr)
        sys.exit(1)

print(f"\\n🚀 Starting AI Video Dubbing pipeline on GPU for: {{video_path}}")

dubber = EnhancedVideoDubbing(
    video_path=video_path,
    source_lang={repr(source_language)},
    target_lang={repr(target_language)},
    whisper_model={repr(whisper_model)},
    voice_quality={repr(voice_quality)},
    enable_lipsync={repr(enable_lipsync)},
    preserve_bg={repr(preserve_background_audio)},
    hf_token=os.getenv("HF_TOKEN"),
    groq_token=os.getenv("Groq_TOKEN"),
)
dubber.process()
"""

proc = subprocess.Popen(
    [sys.executable, "-u", "-c", runner_code],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Processing failed (exit code {proc.returncode}) - check log above.")

# Show and download the result
result_path = "results/dubbed_video.mp4"
if os.path.exists(result_path):
    print(f"\n✅ Done! Output: {result_path}")
    from IPython.display import Video, display
    display(Video(result_path, embed=True, width=640))
    from google.colab import files
    files.download(result_path)
else:
    print("❌ Processing finished but the output video wasn't found - check the log above for errors.")